In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as sns
import seaborn as snS

import nltk
from sentence_transformers import SentenceTransformer

import re
from pathlib import Path
from zipfile import ZipFile


In [3]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to C:\Users\MY
[nltk_data]     PC\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [4]:
DATAFILES_PATH = "../Data"

In [5]:
list(Path(DATAFILES_PATH).rglob("*"))

[WindowsPath('../Data/Cleaned_articles'),
 WindowsPath('../Data/Guidlines'),
 WindowsPath('../Data/Hospital_docs'),
 WindowsPath('../Data/Reports'),
 WindowsPath('../Data/Research_papers'),
 WindowsPath('../Data/Tables'),
 WindowsPath('../Data/Cleaned_articles/archive.zip'),
 WindowsPath('../Data/Cleaned_articles/articles'),
 WindowsPath('../Data/Cleaned_articles/articles/1.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/10.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/100.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1000.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1001.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1002.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1003.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1004.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1005.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1006.txt'),
 WindowsPath('../Data/Cleaned_articles/articles/1007.txt'),
 WindowsPath('../Da

In [6]:
def get_file_list(data_path=DATAFILES_PATH):
    data_path = Path(data_path)
    dirs = []
    files = []

    for item in data_path.rglob("*"):
        if item.is_dir():
            dirs.append(item)
        else:
            files.append(item)

    all_files = [str(i) for i in files if not (Path(i).is_dir() or str(i).endswith(".zip"))]
    pdf_files = [str(i) for i in all_files if str(i).endswith(".pdf")]
    txt_files = [str(i) for i in all_files if str(i).endswith(".txt")]
    csv_files = [str(i) for i in all_files if str(i).endswith(".csv")]
    
    return all_files, pdf_files, txt_files, csv_files

In [7]:
all_files, pdf_files, txt_files, csv_files = get_file_list()

In [8]:
csv_files

['..\\Data\\Tables\\Diseases_Symptoms.csv',
 '..\\Data\\Tables\\medquad.csv',
 '..\\Data\\Tables\\Symptom2Disease.csv']

In [ ]:
def clean_txt_file(txt_file):
    doc_name = Path(txt_file).stem
    doc_type = Path(txt_file).suffix

    with open(txt_file, 'r', encoding='utf-8', errors='ignore') as file:
        text = file.read()
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Tokenize and remove stop words
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    cleaned_text = ' '.join(tokens)
    return cleaned_text

In [39]:
def clean_csv_file(csv_file):
    doc_name = Path(csv_file).stem
    doc_type = Path(csv_file).suffix
    
    df = pd.read_csv(csv_file)

    df.dropna(inplace=True)  # drop rows with any NaN values
    df = df.select_dtypes(include=[object])  # select only string columns

    def clean_text(row):
        new_list = []
        for col in row.index:  # iterate through columns
            new_list.append(f"{col.capitalize()}: {row[col]}")  
        return ". ".join(new_list)
    

    return doc_type, doc_name, df.apply(clean_text, axis=1).to_list()

In [38]:
clean_csv_file(csv_files[1])[0]

'.csv'

In [ ]:
pd.read_csv(csv_files[0]).head()


,Code,Name,Symptoms,Treatments
0,1,Panic disorder,"Palpitations, Sweating, Trembling, Shortness o...","Antidepressant medications, Cognitive Behavior..."
1,2,Vocal cord polyp,"Hoarseness, Vocal Changes, Vocal Fatigue","Voice Rest, Speech Therapy, Surgical Removal"
2,3,Turner syndrome,"Short stature, Gonadal dysgenesis, Webbed neck...","Growth hormone therapy, Estrogen replacement t..."
3,4,Cryptorchidism,"Absence or undescended testicle(s), empty scro...",Observation and monitoring (in cases of mild o...
4,5,Ethylene glycol poisoning-1,"Nausea, vomiting, abdominal pain, General mala...","Supportive Measures, Gastric Decontamination, ..."


In [ ]:
df_trial = pd.read_csv(csv_files[2])
df_trial.head()

df_trial = df_trial.select_dtypes(include=[object]).head()

def clean_text(row):
    new_list = []
    for col in row.index:  # iterate through columns
        new_list.append(f"{col.capitalize()}: {row[col]}")  
    return ". ".join(new_list)

df_trial.apply(clean_text, axis=1).to_list()



['Label: Psoriasis. Text: I have been experiencing a skin rash on my arms, legs, and torso for the past few weeks. It is red, itchy, and covered in dry, scaly patches.',
 'Label: Psoriasis. Text: My skin has been peeling, especially on my knees, elbows, and scalp. This peeling is often accompanied by a burning or stinging sensation.',
 'Label: Psoriasis. Text: I have been experiencing joint pain in my fingers, wrists, and knees. The pain is often achy and throbbing, and it gets worse when I move my joints.',
 'Label: Psoriasis. Text: There is a silver like dusting on my skin, especially on my lower back and scalp. This dusting is made up of small scales that flake off easily when I scratch them.',
 'Label: Psoriasis. Text: My nails have small dents or pits in them, and they often feel inflammatory and tender to the touch. Even there are minor rashes on my arms.']

In [16]:
pd.read_csv(csv_files[1]).head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [ ]:
pd.read_csv(csv_files[2]).head()

,Unnamed: 0,label,text
0,0,Psoriasis,I have been experiencing a skin rash on my arm...
1,1,Psoriasis,"My skin has been peeling, especially on my kne..."
2,2,Psoriasis,I have been experiencing joint pain in my fing...
3,3,Psoriasis,"There is a silver like dusting on my skin, esp..."
4,4,Psoriasis,"My nails have small dents or pits in them, and..."


In [24]:
text_data = []
for i in txt_files:
    with open(i, 'r', encoding='utf-8') as file:
        text_data.append(file.read())

In [ ]:
model = SentenceTransformer('all-mpnet-base-v2')

model.encode(text_data)